# JP-1 G0 2回目（変更1：float32・先頭1回・GPU・Sarashina2.2）

凍結した計画 `notes/jp1_sarashina_gate.md`（凍結 3198152）のゲート G0 を、**変更1**（`notes/jp1_amendment1.md`、2026-10-15 12:25 JST にオーナーが承認）のとおり、最初からやり直します。1回目（bfloat16）は G1 で止まりました（`notes/jp1_lock_pre_g0.md` の追記）。

- 1回目と同じ順番です：版の確かめ → 付録C・D → GPU の計算の照合 → **G0 用の16問**の正解率（float32 で測り直し、0.20 未満なら 3B に1回だけ切り替え）→ 本番前の確認 → 時間と CU の見込み。
- 変更1で変わるところ：
  - モデルは **float32** で、TF32 は使いません。推論の前に確かめます。
  - 引き直しは**先頭を1回だけ計算する**新しいやり方にします。
  - 本番前の確認に **B2**（新しいやり方を2回回して同じか）、**B3**（キャッシュなしの計算と 0.001 以内か）、**M1**（作業2つを同時に、いちばん長い列で、空き 1.0 GB 以上か）を足します。
  - 作業は**2つまで**です。
- どれかが不合格なら `JP1_STOPPED_PRECHECK` などで止まります。合格には全部が要ります。
- 問題の中身・モデルの答え・トークンは、画面にもファイルにも出しません（数・ID・ハッシュだけ）。

**使い方**：ランタイムは **GPU（L4）**。シークレットは `GITHUB_TOKEN`。「すべてのセルを実行」で **30〜50 分くらい**（3B に切り替えるともう少し）。結果は Drive の `DECISION_POINTS/jp1/G0_RUN2/`（1回目の `jp1/G0/` はそのまま残します）。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

In [ ]:
# Cell 1 — Drive, GPU, exact head, notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'JP-1 G0 2回目（変更1・float32・先頭1回・B2/B3/M1つき・GPU・Sarashina）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - JP-1 G0 run2', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'jp1_g0r2'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), "GPU のランタイム（L4）にしてね"
GPU_NAME = gpu.stdout.strip().split(",")[0]
EXPECTED_HEAD = "60e611bb585011a35138d07e1b10219199039c7f"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
D = Path("/content/drive/MyDrive/DECISION_POINTS/jp1/G0_RUN2")   # run 2 (amendment 1); run 1 stays in jp1/G0
D.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))
STARTED = time.time()
print("GPU:", GPU_NAME, "| head:", head)

In [ ]:
# Cell 2 — pinned packages and version checks (before any inference).  Stops with JP1_STOPPED_VERSION_MISMATCH.
from pork import jp1 as P
PINS = [f"{k}=={v}" for k, v in P.PACKAGES.items()] + ["protobuf"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
import importlib.metadata as md
CHECKS = {}
CHECKS["packages"] = {k: md.version(k) for k in P.PACKAGES}
CHECKS["torch"] = md.version("torch")
CHECKS["lock_file_present"] = (REPO / "notes/jp1_lock_pre_g0.md").exists()
CHECKS["gate_doc_sha256_ok"] = P.gate_doc_sha256(REPO) == P.GATE_DOC_SHA256
CHECKS["amendment1_present"] = (REPO / P.AMENDMENT1_RELPATH).exists()
import torch as torch_mod
CHECKS["precision_flags"] = P.precision_flags(torch_mod)                     # D1, before any inference
CHECKS["torch_cuda"] = str(torch_mod.version.cuda)
CHECKS["gpu"] = GPU_NAME
CHECKS["model_dtypes"] = {k: v["dtype"] for k, v in P.MODELS.items()}
import nltk
nltk.download("stopwords", quiet=True)
SW_PATH = Path(nltk.data.find("corpora/stopwords/english"))
CHECKS["stopwords_sha256"] = hashlib.sha256(SW_PATH.read_bytes()).hexdigest()
STOPWORDS = frozenset(w.strip().lower() for w in SW_PATH.read_text(encoding="utf-8").splitlines() if w.strip())
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
MODEL_DIRS = {}
for k, spec in P.MODELS.items():
    MODEL_DIRS[k] = Path(f"/content/sarashina_{k}")
    snapshot_download(spec["id"], revision=spec["revision"], local_dir=str(MODEL_DIRS[k]), token=False,
                      allow_patterns=["*.json", "*.safetensors", "tokenizer*", "*.model"])
    tok = AutoTokenizer.from_pretrained(str(MODEL_DIRS[k]))
    vocab = sorted(tok.get_vocab().items())
    CHECKS[f"tokenizer_{k}"] = {
        "vocab_sha256_ok": hashlib.sha256(json.dumps(vocab, ensure_ascii=False).encode("utf-8")).hexdigest() == P.TOKENIZER_VOCAB_SHA256,
        "chat_template_sha256_ok": hashlib.sha256((tok.chat_template or "").encode("utf-8")).hexdigest() == P.CHAT_TEMPLATE_SHA256}
ROWS = {L: P.load_rows(L) for L in ("ja", "en")}
CHECKS["pairing"] = P.check_pairing(ROWS["ja"], ROWS["en"])
CHECKS["indices_sha256"] = P.indices_sha256()
ITEMS = {L: P.items(ROWS[L], "main") for L in ROWS}
G0_ITEMS = {L: P.items(ROWS[L], "g0") for L in ROWS}
CHECKS["items_sha256"] = {L: P.items_sha256(ITEMS[L]) for L in ITEMS}
CHECKS["g0_items_sha256"] = {L: P.items_sha256(G0_ITEMS[L]) for L in G0_ITEMS}
VERSION_OK = (CHECKS["lock_file_present"] and CHECKS["gate_doc_sha256_ok"]
              and CHECKS["packages"] == P.PACKAGES and CHECKS["torch"].split("+")[0] == P.TORCH_BASE_VERSION
              and CHECKS["stopwords_sha256"] == P.STOPWORDS_SHA256
              and all(all(CHECKS[f"tokenizer_{k}"].values()) for k in P.MODELS) and CHECKS["pairing"]["pass"]
              and CHECKS["amendment1_present"] and P.precision_ok(CHECKS["precision_flags"])
              and all(v == "float32" for v in CHECKS["model_dtypes"].values()))
# the longest prompt (tokens only; 1B and 3B share the tokenizer) -> the longest sequence for M1
from pork import kv_cached_backend as kv
_tk = AutoTokenizer.from_pretrained(str(MODEL_DIRS["1b"]))
CHECKS["max_prompt_tokens"] = {L: max(len(kv.chat_prompt_ids(_tk, P.user_message(L, it["question"])))
                                       for it in ITEMS[L] + G0_ITEMS[L]) for L in ITEMS}
LONGEST = max(CHECKS["max_prompt_tokens"].values()) + P.MAX_NEW_TOKENS
CHECKS["longest_sequence"] = LONGEST
del _tk
print(json.dumps({k: v for k, v in CHECKS.items() if k not in ("items_sha256", "g0_items_sha256")}, indent=1, ensure_ascii=False))
def write_json(name, obj):
    p = D / name
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=1, sort_keys=True, default=str), encoding="utf-8")
    return hashlib.sha256(p.read_bytes()).hexdigest()
write_json("G0_CHECKS.json", {"head": EXPECTED_HEAD, "checks": CHECKS, "version_ok": VERSION_OK, "gpu": GPU_NAME})
assert VERSION_OK, P.STOP_VERSION + " — 推論の前に止めました。Claude に見せてね。"
print("VERSIONS OK")

In [ ]:
# Cell 3 — appendix C / D recheck with the pinned tools (the same checks as C0).
from pork import jp1_text as J
from pork import dp1_forks as dp
import fugashi
TAGGER = fugashi.Tagger()
TOK = AutoTokenizer.from_pretrained(str(MODEL_DIRS["1b"]))
dec = lambda ids: TOK.decode(list(ids), skip_special_tokens=True)
D_ROWS = J.check_appendix_d()
rows = []
for cid, lang, sent, specs in J.APPENDIX_C:
    ids = TOK(sent, add_special_tokens=False)["input_ids"]
    T, words, toks = J.classify(lang, ids, dec, stopwords=STOPWORDS, tagger=TAGGER)
    rows.append(J.check_sentence(cid, lang, sent, specs, T, words, toks))
C_VERDICT = J.appendix_c_verdict(rows)
A_OK = []
for i, lang, text, val, reason, start in J.APPENDIX_D:
    if start is None:
        continue
    ids = TOK(text, add_special_tokens=False)["input_ids"]
    T, spans = J.token_spans(ids, dec)
    A_OK.append(T == text and dp.answer_first_token(ids, dec, start) == J.answer_token_ref(spans, start))
RECHECK = {"appendix_d_ok": all(x["ok"] for x in D_ROWS), "appendix_d": len(D_ROWS), "appendix_c": C_VERDICT,
           "answer_token_ok": all(A_OK), "answer_token_checked": len(A_OK)}
write_json("G0_RECHECK.json", RECHECK)
print(json.dumps(RECHECK, indent=1))
assert RECHECK["appendix_d_ok"] and RECHECK["answer_token_ok"], P.STOP_READOUT
assert C_VERDICT["pass"], P.STOP_CATEGORY
print("APPENDIX C / D OK")

In [ ]:
# Cell 4 — GPU vs plain-Python parity on synthetic logits (vocab 2,000 and 102,400): per-step signals, pure and alt draws.
import random, struct
import torch as torch_mod
from pork import pork_t_v1 as pt
from pork import q4b_samplers as qb
Bg = pt.backend("torch", "cuda")
def f32(x):
    return struct.unpack("f", struct.pack("f", x))[0]
def synth(rng, vocab):
    xs = [f32(rng.gauss(0.0, rng.choice((1.0, 2.5, 4.0)))) for _ in range(vocab)]
    for i in range(0, vocab, 97):
        xs[i] = xs[0]
    if rng.random() < 0.3:
        j = rng.randrange(vocab); xs[j] = f32(xs[j] + 12.0)
    if rng.random() < 0.2:
        xs[1] = xs[0] = max(xs)
    return xs
rng = random.Random(202610151001)
names = ("SIGNALS", "PURE", "ALT")
total = {k: 0 for k in names}; mism = {k: 0 for k in names}
for vocab, steps in ((2000, 150), (102400, 6)):
    mg = mr = None
    for step in range(steps):
        xs = synth(rng, vocab)
        g = torch_mod.tensor(xs, dtype=torch_mod.float32, device="cuda")
        a = dp.step_signals(Bg, g, mg); b = dp.ref_step_signals(xs, mr)
        total["SIGNALS"] += 1
        if ((a["top1"], a["top2"]) != (b["top1"], b["top2"]) or abs(a["p1"] - b["p1"]) > 1e-9
                or abs(a["p2"] - b["p2"]) > 1e-9 or abs(a["entropy"] - b["entropy"]) > 1e-9):
            mism["SIGNALS"] += 1
        mg, mr = a["mu_after"], b["mu_after"]
        u = rng.random()
        total["PURE"] += 1
        mism["PURE"] += int(tuple(int(v) for v in qb.pure_sample(Bg, g, u, P.TEMPERATURE))
                            != tuple(int(v) for v in qb.ref_pure(xs, u, P.TEMPERATURE)))
        ex = b["top1"] if step % 2 == 0 else rng.randrange(vocab)
        u2 = rng.random() if step % 7 else 0.9999999999
        aa, ab = P.alt_sample(Bg, g, u2, P.TEMPERATURE, ex), P.ref_alt(xs, u2, P.TEMPERATURE, ex)
        total["ALT"] += 1
        mism["ALT"] += int(aa != ab or aa == ex)
PARITY = {"total": total, "mismatches": mism, "pass": all(v == 0 for v in mism.values()), "gpu": GPU_NAME}
write_json("G0_PARITY.json", PARITY)
print(json.dumps(PARITY, indent=1))
assert PARITY["pass"], "GPU parity FAILED — Claude に見せてね"
print("PARITY OK")

In [ ]:
# Cell 5 — G0 accuracy on the 16 G0 items (never used in the main run): greedy base path, JA and EN.
# 1B first; below 0.20 in either language -> 3B once, both languages again; still below -> JP1_STOPPED_LOW_ACCURACY.
from pork import kv_cached_backend as kv
from pork import jp1_runner as rr
from pork.ct1_runner import torch_fns
from pork.dp1_forks_runner import generate_base
Bt = pt.backend("torch", "cuda")

def load(k):
    torch_, tok, model = rr.load_model(str(MODEL_DIRS[k]), None, "cuda", P.MODELS[k])
    assert P.model_ok(model, P.MODELS[k]), P.STOP_VERSION + " — dtype / attention（変更1：float32・eager）"
    return tok, model

def g0_run(k):
    tok, model = load(k)
    eos = kv.eos_ids(tok, model)
    single, batch = torch_fns(model, torch_mod, "cuda")
    dec = lambda ids: tok.decode(list(ids), skip_special_tokens=True)
    res = {}
    for L in ("ja", "en"):
        per = []
        for it in G0_ITEMS[L]:
            prompt = tuple(kv.chat_prompt_ids(tok, P.user_message(L, it["question"])))
            g = generate_base(Bt, single(), prompt, eos, P.MAX_NEW_TOKENS)
            b = P.base_record(L, g["token_ids"], g["eos"], g["steps"], dec, it["answer_number"],
                              stopwords=STOPWORDS, tagger=TAGGER)
            per.append({"index": it["index"], "correct": b["correct"], "reason": b["readout_reason"],
                        "tokens": len(b["token_ids"]), "eos": b["eos"], "answer_token": b["answer_token"],
                        "unclassified": sum(c is None for c in b["category"])})
        n = len(per)
        res[L] = {"accuracy": sum(p["correct"] for p in per) / n, "correct": sum(p["correct"] for p in per), "n": n,
                  "reasons": {r: sum(p["reason"] == r for p in per) for r in sorted({p["reason"] for p in per})},
                  "median_tokens": sorted(p["tokens"] for p in per)[n // 2], "per_item": per}
    return res, tok, model, eos

t0 = time.time()
G0 = {"1b": None, "3b": None, "chosen": None, "stopped": None}
G0["1b"], TOKC, MODEL, EOS = g0_run("1b")
print({L: (G0["1b"][L]["correct"], G0["1b"][L]["n"], G0["1b"][L]["reasons"]) for L in ("ja", "en")})
if min(G0["1b"][L]["accuracy"] for L in ("ja", "en")) < P.G0_SWITCH_BELOW:
    del MODEL; torch_mod.cuda.empty_cache()
    G0["3b"], TOKC, MODEL, EOS = g0_run("3b")
    print("3B:", {L: (G0["3b"][L]["correct"], G0["3b"][L]["n"]) for L in ("ja", "en")})
    if min(G0["3b"][L]["accuracy"] for L in ("ja", "en")) < P.G0_SWITCH_BELOW:
        G0["stopped"] = P.STOP_LOW_ACCURACY
    else:
        G0["chosen"] = "3b"
else:
    G0["chosen"] = "1b"
G0["seconds"] = time.time() - t0
write_json("G0_ACCURACY.json", G0)
assert G0["stopped"] is None, P.STOP_LOW_ACCURACY + " — 3B でも正解率が 0.20 未満でした。Claude に見せてね。"
print("G0 model:", G0["chosen"])

In [ ]:
# Cell 6 — E2 / G1 / B1 with the chosen model on harmless prompts (no benchmark text).
from pork import quality_pork_q8t as q8
single, _batch_old = torch_fns(MODEL, torch_mod, "cuda")
from pork import jp1_shared_prefix as SP
batch = SP.torch_shared_prefix_batch(MODEL, torch_mod, "cuda")     # amendment 1 (C2): B1, B2, B3, timing use it
e2, g1 = [], []
for name, text in P.CHECK_PROMPTS:
    ids = kv.chat_prompt_ids(TOKC, text)
    step = single()
    seq, past, new, worst, agree = list(ids), None, tuple(ids), 0.0, True
    for _ in range(P.E2_STEPS):
        a, past = step(new, past)
        with torch_mod.no_grad():
            bfull = MODEL(input_ids=torch_mod.tensor([seq], device="cuda"), use_cache=False).logits[0, -1].float()
        worst = max(worst, float((a - bfull).abs().max()))
        ta, tb = int(a.argmax()), int(bfull.argmax())
        agree = agree and ta == tb
        seq.append(tb); new = (tb,)
    e2.append({"prompt": name, "max_abs_logit_diff": worst, "argmax_agree": agree})
    ref_tokens, seq = [], list(ids)
    for _ in range(P.G1_STEPS):
        with torch_mod.no_grad():
            tok_ = q8._greedy([float(v) for v in MODEL(input_ids=torch_mod.tensor([seq], device="cuda"),
                                                        use_cache=False).logits[0, -1].float().tolist()])
        ref_tokens.append(tok_); seq.append(tok_)
        if tok_ in set(EOS):
            break
    base = generate_base(Bt, single(), ids, EOS, len(ref_tokens))["token_ids"]
    g1.append({"prompt": name, "steps": len(ref_tokens), "identical": base == ref_tokens})
ids = kv.chat_prompt_ids(TOKC, P.CHECK_PROMPTS[0][1])
_bs, _ss = batch(), single()
bp, sp, bn, sn = None, None, [tuple(ids)] * P.K, tuple(ids)
b1_agree = 0
for _ in range(32):
    bl, bp = _bs(bn, bp); sl, sp = _ss(sn, sp)
    t_ = int(sl.argmax()); b1_agree += int(int(bl[0].argmax()) == t_)
    bn, sn = [(t_,)] * P.K, (t_,)
del _bs, _ss, bp, sp, bl, sl
g1.append({"prompt": "B1_batched_vs_single", "steps": 32, "identical": b1_agree >= P.B1_MIN_AGREE, "agree_steps": b1_agree})
# --- amendment 1: B2 (D5) and B3 (D3), then the prechecks verdict (JP1_STOPPED_PRECHECK) ---
from pork import jp1_runner as rr
from pork.ct1_forks_replacement import generate_batch
_ids_ja = tuple(kv.chat_prompt_ids(TOKC, P.CHECK_PROMPTS[0][1]))
_ids_en = tuple(kv.chat_prompt_ids(TOKC, P.CHECK_PROMPTS[1][1]))
_path_ja = generate_base(Bt, single(), _ids_ja, EOS, max(P.B3_CUTS) + P.B3_STEPS + 1)["token_ids"]
_path_en = generate_base(Bt, single(), _ids_en, EOS, 65)["token_ids"]

# B2: the new step twice, same prefix and streams, B2_STEPS steps; all K rows identical, and the rows must have branched
B2 = {"tries": []}
for _name, _pre in (("check_ja_cut64", _ids_ja + tuple(_path_ja[:64])), ("check_en_cut64", _ids_en + tuple(_path_en[:64]))):
    _streams = [P.stream("free", 2, 0, j) for j in range(P.K)]
    _a = generate_batch(Bt, batch(), _pre, _streams, EOS, P.TEMPERATURE, P.B2_STEPS)
    _b = generate_batch(Bt, batch(), _pre, _streams, EOS, P.TEMPERATURE, P.B2_STEPS)
    _distinct = len({tuple(o["token_ids"]) for o in _a})
    B2["tries"].append({"prefix": _name, "prefix_tokens": len(_pre), "rows_identical": sum(x == y for x, y in zip(_a, _b)),
                        "rows": P.K, "distinct_paths": _distinct,
                        "sha256_run1": hashlib.sha256(json.dumps([o["token_ids"] for o in _a]).encode()).hexdigest(),
                        "sha256_run2": hashlib.sha256(json.dumps([o["token_ids"] for o in _b]).encode()).hexdigest()})
    if _distinct > 1:
        break
_last = B2["tries"][-1]
B2["branched"] = _last["distinct_paths"] > 1
B2["pass"] = B2["branched"] and _last["rows_identical"] == _last["rows"]
del _a, _b

# B3: new-step logits against a full no-cache forward of the same row, at three prefix lengths, kinds free and alt
def _full_last(seq):
    with torch_mod.no_grad():
        return MODEL(input_ids=torch_mod.tensor([list(seq)], device="cuda"), use_cache=False).logits[0, -1].float()
def _recording(step_fn):
    rec, n = {}, [0]
    def step(rows, past):
        lg, past = step_fn(rows, past)
        if n[0] in P.B3_CHECK_STEPS:
            rec[n[0]] = {r: lg[r].detach().clone() for r in P.B3_ROWS}
        n[0] += 1
        return lg, past
    return step, rec
B3 = {"checks": []}
for _cut in P.B3_CUTS:
    if _cut >= len(_path_ja):                 # the greedy path ended earlier: recorded, not compared
        B3["checks"].append({"cut": _cut, "skipped_short_path": True, "compared": 0, "max_abs_logit_diff": 0.0})
        continue
    _pre = _ids_ja + tuple(_path_ja[:_cut])
    for _kind in ("free", "alt"):
        _streams = [P.stream(_kind, 3, _cut, j) for j in range(P.K)]
        _draw = rr.alt_draw(Bt, int(_path_ja[_cut])) if _kind == "alt" else None
        _step, _rec = _recording(batch())
        _outs = generate_batch(Bt, _step, _pre, _streams, EOS, P.TEMPERATURE, P.B3_STEPS, draw=_draw)
        _worst, _n = 0.0, 0
        for _s, _rows in _rec.items():
            for _r, _lg in _rows.items():
                _toks = _outs[_r]["token_ids"]
                if len(_toks) <= _s:          # the row had finished; its later inputs are padding
                    continue
                _worst = max(_worst, float((_lg - _full_last(_pre + tuple(_toks[:_s]))).abs().max()))
                _n += 1
        B3["checks"].append({"cut": _cut, "kind": _kind, "prefix_tokens": len(_pre), "compared": _n, "max_abs_logit_diff": _worst})
        del _rec, _outs, _step
B3["max_abs_logit_diff"] = max(c["max_abs_logit_diff"] for c in B3["checks"])
B3["compared"] = sum(c["compared"] for c in B3["checks"])
B3["pass"] = B3["compared"] > 0 and B3["max_abs_logit_diff"] <= P.B3_TOL

PRE = {"E2": e2, "G1_B1": g1, "B2": B2, "B3": B3,
       "pass": (all(r["argmax_agree"] for r in e2) and all(r["identical"] for r in g1) and B2["pass"] and B3["pass"])}
write_json("G0_PRECHECKS.json", PRE)
print(json.dumps({"E2": [r["argmax_agree"] for r in e2], "G1_B1": [r["identical"] for r in g1],
                  "B2": {"pass": B2["pass"], "rows_identical": _last["rows_identical"], "distinct_paths": _last["distinct_paths"]},
                  "B3": {"pass": B3["pass"], "max_abs_logit_diff": B3["max_abs_logit_diff"], "compared": B3["compared"]}}, indent=1))
assert PRE["pass"], P.STOP_PRECHECK + " — E2 / G1 / B1 / B2 / B3 のどれかが不合格です。Claude に見せてね"
print("E2 / G1 / B1 / B2 / B3 OK")

In [ ]:
# Cell 7 — timing smoke and the stop lines: per-notebook estimate x 2.7 <= 14 h; whole-study CU estimate <= 45.
# Base-path length per language: the median of the G0 items' base paths (numbers only).
from pork.ct1_forks_replacement import generate_batch
WORKERS = max(1, min(rr.MAX_WORKERS, kv.physical_cores(), int(torch_mod.cuda.get_device_properties(0).total_memory / 1e9 // 10)))
ids = kv.chat_prompt_ids(TOKC, P.CHECK_PROMPTS[0][1])
no_eos = (-1,)
generate_base(Bt, single(), ids, no_eos, 8)
n = P.TIMING_SMOKE_STEPS
per = {}
t0 = time.time(); g = generate_base(Bt, single(), ids, no_eos, n); per["BASE"] = (time.time() - t0) / n
streams = [P.stream("free", 0, 0, j) for j in range(P.K)]
generate_batch(Bt, batch(), ids, streams, no_eos, P.TEMPERATURE, 4)
t0 = time.time(); generate_batch(Bt, batch(), ids, streams, no_eos, P.TEMPERATURE, n); per["BATCH32"] = (time.time() - t0) / n
long_ids = tuple(ids) + tuple(g["token_ids"]) * 12
t0 = time.time(); generate_batch(Bt, batch(), long_ids, streams, no_eos, P.TEMPERATURE, 1); per["PREFILL"] = time.time() - t0
KPP = 3 + 1 / P.FREE_EVERY
def per_item(Lb):
    rest = [Lb - t for t in range(0, Lb, P.STRIDE)]
    return Lb * per["BASE"] + sum(rest) * KPP * per["BATCH32"] + len(rest) * KPP * per["PREFILL"]
k = G0["chosen"]
NB = {}
for L in ("ja", "en"):
    Lb = int(G0[k][L]["median_tokens"])
    hours = len(P.HALVES["A"]) * per_item(Lb) / WORKERS / 3600
    NB[L] = {"median_base_tokens": Lb, "typical_hours_per_notebook": hours,
             "calibrated_hours_per_notebook": hours * P.TIMING_FACTOR}
g0_hours = (time.time() - STARTED) / 3600
total_cu = (2 * sum(v["calibrated_hours_per_notebook"] for v in NB.values()) + g0_hours) * P.L4_CU_PER_HOUR
TIMING = {"gpu": GPU_NAME, "workers": WORKERS, "per_step_s": per, "per_notebook": NB, "g0_hours_so_far": g0_hours,
          "cu_per_hour_assumed": P.L4_CU_PER_HOUR, "total_cu_estimate": total_cu,
          "per_notebook_ok": all(v["calibrated_hours_per_notebook"] <= P.TIMING_STOP_HOURS for v in NB.values()),
          "total_cu_ok": total_cu <= P.CU_STOP_TOTAL}
write_json("G0_TIMING.json", TIMING)
print(json.dumps(TIMING, indent=1))
assert TIMING["per_notebook_ok"], "1冊の見込みが 14 時間を超えます — Claude に見せてね"
assert TIMING["total_cu_ok"], "全体の見込みが 45 CU を超えます — Claude に見せてね"
print("TIMING OK")

In [ ]:
# Cell 7b — M1 (amendment 1, D4): free the notebook's GPU model, then two worker processes of the chosen model at once,
# at the longest sequence of the main run (longest prompt + MAX_NEW_TOKENS).  Both must finish and leave >= 1.0 GB free.
import gc
for _n, _v in list(globals().items()):
    if isinstance(_v, (torch_mod.nn.Module, torch_mod.Tensor)):
        del globals()[_n]
for _n in ("single", "batch", "_batch_old", "step", "past", "TOKC"):
    globals().pop(_n, None)
gc.collect(); torch_mod.cuda.empty_cache()
KERNEL_ALLOCATED_GB = torch_mod.cuda.memory_allocated() / 1e9
BAR = Path("/content/jp1_m1_barrier")
if BAR.exists():
    shutil.rmtree(BAR)
BAR.mkdir()
_k = G0["chosen"]
_procs = {}
for _tag in ("w0", "w1"):
    _procs[_tag] = subprocess.Popen([sys.executable, "-m", "pork.jp1_m1_probe", _k, str(MODEL_DIRS[_k]), str(LONGEST),
                                     f"/content/jp1_m1_{_tag}.json", _tag, str(BAR)],
                                    cwd=str(REPO), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _p in _procs.values():
    _p.wait(timeout=1800)
M1W = {}
for _tag, _p in _procs.items():
    _f = Path(f"/content/jp1_m1_{_tag}.json")
    M1W[_tag] = json.loads(_f.read_text()) if _f.exists() else {"tag": _tag, "ok": False, "error": "no output"}
    M1W[_tag]["returncode"] = _p.returncode
_free = min(v.get("device_free_gb_while_both_hold", 0.0) for v in M1W.values())
M1 = {"model": _k, "longest_sequence": LONGEST, "workers": M1W, "gpu": GPU_NAME, "kernel_allocated_gb_before": KERNEL_ALLOCATED_GB,
      "min_free_gb_while_both_hold": _free, "min_free_line_gb": P.M1_MIN_FREE_GB,
      "pass": all(v.get("ok") and v["returncode"] == 0 for v in M1W.values()) and _free >= P.M1_MIN_FREE_GB}
write_json("G0_M1.json", M1)
print(json.dumps({"pass": M1["pass"], "model": _k, "longest_sequence": LONGEST, "min_free_gb": round(_free, 2),
                  "peak_reserved_gb": {t: round(v.get("peak_reserved_gb", -1), 2) for t, v in M1W.items()},
                  "errors": {t: v.get("error") for t, v in M1W.items() if v.get("error")}}, indent=1))
assert M1["pass"], P.STOP_PRECHECK + " — M1（作業2つのメモリ）が不合格です。Claude に見せてね"
print("M1 OK")

In [ ]:
# Cell 8 — G0 summary (numbers only), then notify and disconnect.
SUMMARY = {"schema": "jp1_g0_run2_v1", "run": 2, "amendment": P.AMENDMENT1_RELPATH, "head": EXPECTED_HEAD, "finished_jst": dt.datetime.now(JST).isoformat(timespec="seconds"),
           "gpu": GPU_NAME, "checks": CHECKS, "recheck": RECHECK, "parity_pass": PARITY["pass"],
           "g0": {k2: ({L: {kk: vv for kk, vv in G0[k2][L].items() if kk != "per_item"} for L in ("ja", "en")}
                      if G0[k2] else None) for k2 in ("1b", "3b")},
           "chosen_model": G0["chosen"], "prechecks_pass": PRE["pass"], "timing": TIMING,
           "amendment1": {"B2": B2, "B3": B3, "M1": M1, "precision_flags": CHECKS["precision_flags"],
                          "torch": CHECKS["torch"], "cuda": CHECKS["torch_cuda"], "gpu": GPU_NAME,
                          "max_gpu_workers": P.MAX_GPU_WORKERS}}
sha_ = write_json("G0_SUMMARY.json", SUMMARY)
print("G0_SUMMARY.json sha256:", sha_)
print("G0（2回目）合格：モデル", G0["chosen"], "| 正解率", {L: G0[G0['chosen']][L]['correct'] for L in ('ja', 'en')}, "/ 16")
notify("G0（2回目）終わりました：モデル " + str(G0["chosen"]), tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()